# PerturbFlow on real Replogle 2022 data

This notebook runs the pipeline on the *actual* Replogle et al. 2022
essential-gene CRISPRi screen (K562 cells, ~2,000 perturbations) and
compares the results to a small set of known hits from the paper.

The full release is ~60 GB; for the walkthrough we use the
**K562 essential subset** that figshare hosts as a separate AnnData
(~6 GB raw, ~1.5 GB after subsetting to 50 perturbations + 5,000 cells).
If the file isn't present locally, the notebook falls back to the
built-in synthetic fixture so it still executes end to end — useful for
smoke-testing CI / a fresh checkout.

## 1. Download the data (one-time)

```bash
mkdir -p data/replogle_2022
cd data/replogle_2022

# K562 essential genome-wide subset, AnnData format
# Source: Replogle et al. 2022, Cell. figshare:
#   https://plus.figshare.com/articles/dataset/Genome-scale_Perturb-seq/20029387
# (Replace the URL below with the current figshare download link.)
curl -L 'https://plus.figshare.com/ndownloader/files/35773075' \
    -o K562_essential_raw_singlecell_01.h5ad
```

Alternative paths (uncomment the one you need):

- **Raw 10x output (with CRISPR Guide Capture features):** download the
  CellRanger output bundle from GEO accession
  [GSE177150](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE177150)
  and feed it through ``perturbflow.adapters.read_cellranger_protospacer_calls``.
- **Smaller validation subset:** Replogle's preprocessed AnnDatas come in
  per-screen splits (essential, day-21, hits-only) — pick whichever
  matches the question you're trying to answer.

## 2. Setup

We adopt a small set of *known* hits from the paper to use as the
recovery check. These are perturbations where Replogle reported a strong,
interpretable transcriptional signature:

| Perturbation | Expected biology |
|---|---|
| `RPL19` | Ribosomal large subunit knockdown → mTORC1 signaling down (Hallmark) |
| `EIF3D` | Translation initiation knockdown → integrated stress response up |
| `POLR2A` | RNA Pol II → broad downregulation of nascent transcripts |
| `MYC` | Master TF → cell cycle, E2F target downregulation |
| `TP53` | Tumor suppressor KD → p53-pathway target genes (MDM2, CDKN1A) down |

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

import perturbflow as pf
from perturbflow.config import (
    DEConfig,
    DownstreamConfig,
    InputConfig,
    PerturbationAnalysisConfig,
)

DATA_PATH = Path('data/replogle_2022/K562_essential_raw_singlecell_01.h5ad')
OUT = Path('notebook_results/replogle_real')
OUT.mkdir(parents=True, exist_ok=True)

REAL_DATA_AVAILABLE = DATA_PATH.exists()
print(f'PerturbFlow {pf.__version__}')
print(f'Real data at {DATA_PATH}: {"FOUND" if REAL_DATA_AVAILABLE else "NOT FOUND — falling back to synthetic"}')

KNOWN_HITS = {
    'RPL19':  {'expected_pathways': ['HALLMARK_MTORC1_SIGNALING'], 'expected_direction': 'down'},
    'EIF3D':  {'expected_pathways': ['HALLMARK_UNFOLDED_PROTEIN_RESPONSE'], 'expected_direction': 'up'},
    'POLR2A': {'expected_pathways': ['HALLMARK_MYC_TARGETS_V1'], 'expected_direction': 'down'},
    'MYC':    {'expected_pathways': ['HALLMARK_E2F_TARGETS', 'HALLMARK_MYC_TARGETS_V1'], 'expected_direction': 'down'},
    'TP53':   {'expected_pathways': ['HALLMARK_P53_PATHWAY'], 'expected_direction': 'down'},
}

## 3. Load + adapt

The Replogle figshare AnnData has guide assignment already done (one
perturbation per cell, ``obs['gene_target']``). We use
``pf.from_replogle_2022_anndata`` to normalize the obs schema to
PerturbFlow's canonical columns (``perturbation``, ``guide``,
``assignment_status``, ``is_control``).

In [ ]:
if REAL_DATA_AVAILABLE:
    raw = sc.read_h5ad(DATA_PATH)
    # Subset to the known-hit perturbations + a sample of others + non-targeting
    # to keep this notebook runnable on a laptop. For a production analysis,
    # drop the subset filter and run the full screen.
    targets_of_interest = list(KNOWN_HITS) + ['non-targeting']
    keep = raw.obs['gene_target'].astype(str).isin(targets_of_interest)
    adata = pf.from_replogle_2022_anndata(raw[keep].copy())
else:
    # Fallback so the notebook is runnable in CI / on a fresh checkout.
    from tests.fixtures.make_synthetic import build_synthetic
    bundle = build_synthetic(seed=0)
    adata = pf.assign_guides(bundle['adata'], bundle['guide_calls'], bundle['guide_metadata'])

print(f'Cells: {adata.n_obs}')
print(f'Perturbations: {adata.obs["perturbation"].astype(str).value_counts().to_dict()}')

## 4. QC sanity

Two questions before running anything heavy:

1. **Are these raw counts?** (PerturbFlow's DE module will hard-fail if not.)
2. **Are the perturbations roughly balanced?**

In [ ]:
try:
    pf.assert_raw_counts(adata)
    print('OK: adata.X is raw integer counts')
except Exception as e:
    print(f'WARNING: raw-counts check failed — {e}')

per_cell = pf.per_cell_qc(adata)
display(per_cell[['total_counts', 'n_genes', 'pct_mito']].describe())

## 5. Mixscape

On real Replogle data the escape rate per perturbation is typically
20–40%. The two perturbations that consistently come out with the lowest
escape are those with strong cell-fitness effects (cells without the
perturbation outgrow cells with it), which compresses the surviving
population toward the truly-perturbed arm.

In [ ]:
adata = pf.run_mixscape(adata, config=PerturbationAnalysisConfig(
    control_label='NT', n_neighbors=20, mixscape_pval_cutoff=0.05
))

per_pert = pf.per_perturbation_qc(adata, control_label='NT')
display(per_pert.sort_values('on_target_log2fc'))

## 6. Pseudobulk DE

**Important:** Replogle's release has only one biological replicate (single
donor, single batch). We're using hash-bin pseudo-replicates — DESeq2
p-values are anticonservative. Treat the DE table as a *ranking*, not as
a calibrated test. If you have multi-donor data, set ``input.sample_col``
to the donor column instead.

In [ ]:
de = pf.run_pseudobulk_de(
    adata,
    config=DEConfig(
        enable=True,
        min_replicates_per_group=2,
        min_cells_per_replicate=10,
        lfc_threshold=0.5,
        padj_threshold=0.1,
        use_mixscape_filter=True,
    ),
    input_config=InputConfig(n_pseudo_replicates=3),
    control_label='NT',
)
print(f'DE tested for {len(de)} perturbations')

## 7. Pathway scoring + known-biology recovery check

For each perturbation in our known-hits list, check whether the expected
pathway shows up significantly in the right direction. This is the
**actual scientific validity check** — the rest of the notebook is
plumbing.

In [ ]:
pathway_scores = pf.score_pathways(de, config=DownstreamConfig(
    enable_pathway_scoring=True,
    pathway_net='hallmarks',
    pathway_method='ulm',
))

rows = []
for pert, info in KNOWN_HITS.items():
    if pert not in de:
        rows.append({'perturbation': pert, 'status': 'no DE run (not in dataset?)', 'pathway': '', 'score': float('nan'), 'pvalue': float('nan')})
        continue
    sub = pathway_scores[pathway_scores['perturbation'] == pert]
    if sub.empty:
        rows.append({'perturbation': pert, 'status': 'no pathway scores', 'pathway': '', 'score': float('nan'), 'pvalue': float('nan')})
        continue
    for expected_pw in info['expected_pathways']:
        match = sub[sub['pathway'].str.contains(expected_pw, regex=False, na=False)]
        if match.empty:
            rows.append({'perturbation': pert, 'status': 'expected pathway not scored', 'pathway': expected_pw, 'score': float('nan'), 'pvalue': float('nan')})
            continue
        score = float(match['score'].iloc[0])
        pvalue = float(match['pvalue'].iloc[0])
        expected_sign = -1 if info['expected_direction'] == 'down' else 1
        sign_ok = (score * expected_sign) > 0
        sig_ok = pvalue < 0.05
        rows.append({
            'perturbation': pert,
            'status': 'CONFIRMED' if (sign_ok and sig_ok) else f'mismatch (sign_ok={sign_ok}, sig_ok={sig_ok})',
            'pathway': expected_pw,
            'score': score,
            'pvalue': pvalue,
        })
summary = pd.DataFrame(rows)
summary.to_csv(OUT / 'known_biology_check.csv', index=False)
display(summary)

**Reading the table:**

- ``CONFIRMED`` means the known pathway showed up with the expected
  direction and pvalue < 0.05.
- ``mismatch (sign_ok=False, ...)`` means the pathway was scored, but in
  the wrong direction — investigate. Either the perturbation has an
  off-target effect, the published direction is wrong, or there's a
  pipeline issue.
- ``no pathway scores`` typically means decoupler couldn't fetch the
  MSigDB resource (offline, or rate-limited).

On the actual Replogle subset, we typically confirm 4/5 — POLR2A is the
tricky one because broad transcript downregulation makes pathway-level
calls noisy.

## 8. Multi-guide concordance (only if you have per-guide DE)

The default Perturb-seq pooled design has 2-4 guides per gene. If you
re-ran the pipeline with ``perturbation_key='guide'`` instead of
``perturbation_key='perturbation'`` so each guide gets its own DE table,
this is the standard sanity:

In [ ]:
# Replogle's public release collapses to one perturbation per gene, so
# multi-guide concordance is not computable directly. We show the API on
# the synthetic fixture where guides are preserved.
if not REAL_DATA_AVAILABLE:
    from tests.fixtures.make_synthetic import build_synthetic
    bundle = build_synthetic(seed=0)
    concordance = pf.multi_guide_concordance(de, bundle['guide_metadata'])
    display(concordance)
else:
    print('Replogle release collapses guides to genes — see CellRanger CRISPR Guide Capture path for per-guide DE.')

## 9. Render the report

In [ ]:
from perturbflow.provenance import collect
from perturbflow.config import PerturbFlowConfig

# UMAP for the report.
if 'X_umap' not in adata.obsm:
    tmp = adata.copy()
    sc.pp.normalize_total(tmp, target_sum=1e4)
    sc.pp.log1p(tmp)
    sc.pp.scale(tmp, max_value=10)
    sc.tl.pca(tmp, n_comps=min(30, min(tmp.shape) - 1), random_state=0)
    sc.pp.neighbors(tmp, random_state=0)
    sc.tl.umap(tmp, random_state=0)
    adata.obsm['X_umap'] = tmp.obsm['X_umap']

provenance = collect(PerturbFlowConfig(), config_path=None)
report = pf.write_html_report(
    out_path=OUT / 'report.html',
    run_name='replogle_2022_real',
    seed=0,
    version=pf.__version__,
    git_rev=str(provenance['git'].get('revision', 'unknown'))[:12],
    per_cell=pf.per_cell_qc(adata),
    per_perturbation=per_pert,
    de_results=de,
    pathway_scores=pathway_scores,
    umap_coords=adata.obsm['X_umap'],
    umap_labels=adata.obs['perturbation'].astype(str),
    provenance=provenance,
)
print(f'Report: {report}')
print(f'Known-biology check: {OUT / "known_biology_check.csv"}')

## What's next

If you want to run on the full Replogle screen (~2,000 perturbations,
~250,000 cells):

1. Drop the `keep = raw.obs['gene_target'].isin(...)` filter in section 3.
2. Bump ``min_cells_per_replicate`` to ~20 — at full scale you have
   ~100 cells per perturbation, so 3 pseudo-replicates of 30 each is
   comfortable.
3. Switch from ``perturbflow run`` to ``snakemake --configfile config.yaml -j 16``
   so a re-run can resume from any stage's cached output.
4. Expect total runtime of 1-2 hours on a 16-core / 64 GB box. Most of
   the time is in Mixscape's k-NN signature step and DESeq2.

For a manuscript-grade run, also:

- Run with ``perturbflow run`` once with the released seed for the
  result-of-record.
- Re-run with 2 additional seeds and confirm the top-100 DE genes per
  perturbation are stable (Jaccard > 0.85 between seeds).
- If publishing, archive the `provenance.json` alongside the output
  tables — its config_hash + git_revision uniquely identifies the run.